# Forced and Damped Harmonic Oscillator

Explore the motion of a damped oscillator driven by a sinusoidal force. Adjust the driving angular frequency and damping coefficient to compare the total motion with its transient and steady-state parts.

This notebook uses `ipywidgets` controls and standard inline Matplotlib output, following the same pattern as the previous oscillator notebooks.

## Model

The equation of motion is

$$
\frac{d^2x}{dt^2} + 2\gamma\frac{dx}{dt} + \omega_0^2x = \frac{F_0}{m}\cos(\omega_f t).
$$

The natural angular frequency is fixed at $\omega_0=1$, and the force-to-mass ratio is fixed at $F_0/m=1$. The solution is the sum of a transient term, which decays with time, and a steady-state term, which oscillates at the driving frequency.

The initial position and velocity are fixed at $x(0)=5$ and $v(0)=0$. The sliders change the driving frequency $\omega_f$ and damping coefficient $\gamma$.

In addition to position versus time, this notebook plots the **phase-space trajectory**: velocity $v$ versus position $x$. For the driven oscillator, the trajectory generally approaches a closed loop after the transient response decays. A separate comparison lets you examine unforced underdamped, critically damped, and overdamped motion.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

print('NumPy:', np.__version__)
print('Matplotlib:', plt.matplotlib.__version__)
print('ipywidgets:', widgets.__version__)
print('Matplotlib backend:', plt.get_backend())

NumPy: 2.5.4
Matplotlib: 3.11.2
ipywidgets: 8.1.9
Matplotlib backend: module://matplotlib_inline.backend_inline


In [2]:
# Fixed model parameters
x0 = 5.0
v0 = 0.0
F0_over_m = 1.0
w0 = 1.0
t = np.linspace(0.0, 100.0, 5000)

# Interactive controls
sfreq = widgets.FloatSlider(
    value=0.5, min=0.1, max=3.0, step=0.01,
    description='Driving frequency ωf', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
sgam = widgets.FloatSlider(
    value=0.1, min=0.01, max=0.999, step=0.001,
    description='Damping coefficient γ', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

def forced_oscillator(wf, gamma):
    # Underdamped transient response: gamma < w0
    wd = np.sqrt(w0**2 - gamma**2)

    # Steady-state amplitude and phase lag
    amp = F0_over_m / np.sqrt((w0**2 - wf**2)**2 + (2*gamma*wf)**2)
    delta = np.arctan2(2*gamma*wf, w0**2 - wf**2)
    phase = wf*t - delta
    steady = amp * np.cos(phase)
    steady_v = -amp * wf * np.sin(phase)

    # Choose transient coefficients to satisfy the specified initial conditions.
    C = x0 - amp*np.cos(delta)
    D = (v0 + gamma*C - amp*wf*np.sin(delta)) / wd
    decay = np.exp(-gamma*t)
    transient = decay * (C*np.cos(wd*t) + D*np.sin(wd*t))
    transient_v = decay * (
        (-gamma*C + D*wd)*np.cos(wd*t)
        + (-gamma*D - C*wd)*np.sin(wd*t)
    )

    total = transient + steady
    velocity = transient_v + steady_v
    return total, velocity, transient, steady, amp

def plot_forced_oscillator(wf, gamma):
    total, velocity, transient, steady, amp = forced_oscillator(wf, gamma)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Time-domain plot
    axes[0].plot(t, total, color='red', linewidth=1.8, label='Total motion')
    axes[0].plot(t, transient, color='blue', linewidth=1.0, label='Transient part')
    axes[0].plot(t, steady, color='green', linewidth=1.0, label='Steady-state part')
    axes[0].set_xlim(0, 100)
    axes[0].set_xlabel('Time')
    axes[0].set_ylabel('Position, x')
    axes[0].set_title('Position versus time')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Phase-space plot for the driven oscillator
    axes[1].plot(total, velocity, color='purple', linewidth=1.2)
    axes[1].plot([total[0]], [velocity[0]], marker='o', color='black',
                 markersize=6, linestyle='None', label='Initial state')
    axes[1].axhline(0, color='black', linewidth=0.6)
    axes[1].axvline(0, color='black', linewidth=0.6)
    axes[1].set_xlabel('Position, x')
    axes[1].set_ylabel('Velocity, v')
    axes[1].set_title('Phase-space trajectory')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig.suptitle(
        f'Forced and Damped Oscillator: ωf = {wf:.2f}, γ = {gamma:.3f}; '
        f'steady-state amplitude = {amp:.3f}'
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)

plot_output = widgets.interactive_output(
    plot_forced_oscillator,
    {'wf': sfreq, 'gamma': sgam}
)

reset_button = widgets.Button(description='Reset')
def reset_values(button):
    sfreq.value = 0.5
    sgam.value = 0.1
reset_button.on_click(reset_values)

display(widgets.VBox([sfreq, sgam, reset_button]), plot_output)


Output()

## Questions to investigate

1. What happens to the steady-state amplitude as the driving frequency approaches the natural frequency?
2. How does increasing the damping coefficient change the steady-state amplitude near resonance?
3. How does the transient part change as time increases?
4. After the transient has mostly decayed, which part of the solution remains?

## Notes

The transient term is calculated so that the total solution satisfies the initial position and velocity. The steady-state amplitude is not necessarily the largest amplitude of the total motion because the transient and steady-state terms can add or cancel. The vertical axis is allowed to autoscale so large amplitudes near resonance are not clipped.

## Phase diagrams for unforced damping

The driven phase-space plot above uses the same driving-frequency and damping sliders as the time plot. This additional section compares the three damping regimes for the **unforced** oscillator,

$$
\frac{d^2x}{dt^2}+2\gamma\frac{dx}{dt}+\omega_0^2x=0.
$$

Here, $\omega_0=1$. Critical damping occurs at $\gamma=1$; the underdamped case has $\gamma<1$, and the overdamped case has $\gamma>1$. These controls and initial conditions are independent of the driven-oscillator controls above.


In [3]:
# Time array and controls for the unforced comparison
t_free = np.linspace(0.0, 20.0, 1800)

spos = widgets.FloatSlider(
    value=5.0, min=-5.0, max=5.0, step=0.1,
    description='Initial position', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
svel = widgets.FloatSlider(
    value=0.0, min=-5.0, max=5.0, step=0.1,
    description='Initial velocity', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
sgam_under = widgets.FloatSlider(
    value=0.5, min=0.0, max=0.99, step=0.01,
    description='Underdamped γ', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
sgam_over = widgets.FloatSlider(
    value=1.5, min=1.01, max=5.0, step=0.01,
    description='Overdamped γ', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

def phase_solutions(x0, v0, gamma_under, gamma_over):
    # Critical damping: repeated root r = -w0
    critical_x = (x0 + (v0 + w0*x0)*t_free) * np.exp(-w0*t_free)
    critical_v = (v0 - w0*(v0 + w0*x0)*t_free) * np.exp(-w0*t_free)

    # Underdamped case: gamma < w0
    # Avoid division by zero when gamma_under = 0: the formula remains valid.
    wd = np.sqrt(w0**2 - gamma_under**2)
    under_x = np.exp(-gamma_under*t_free) * (
        x0*np.cos(wd*t_free) + ((v0 + gamma_under*x0)/wd)*np.sin(wd*t_free)
    )
    under_v = np.exp(-gamma_under*t_free) * (
        v0*np.cos(wd*t_free)
        - ((gamma_under*v0 + w0**2*x0)/wd)*np.sin(wd*t_free)
    )

    # Overdamped case: two distinct real roots
    q = np.sqrt(gamma_over**2 - w0**2)
    r1 = -gamma_over + q
    r2 = -gamma_over - q
    c1 = (v0 - r2*x0)/(r1 - r2)
    c2 = (r1*x0 - v0)/(r1 - r2)
    over_x = c1*np.exp(r1*t_free) + c2*np.exp(r2*t_free)
    over_v = r1*c1*np.exp(r1*t_free) + r2*c2*np.exp(r2*t_free)

    return (critical_x, critical_v), (under_x, under_v), (over_x, over_v)

def plot_free_phase_diagrams(x0, v0, gamma_under, gamma_over):
    critical, under, over = phase_solutions(x0, v0, gamma_under, gamma_over)

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot(critical[0], critical[1], linewidth=2,
            label='Critical damping (γ = 1)')
    ax.plot(under[0], under[1], linewidth=1.8,
            label=f'Underdamped (γ = {gamma_under:.2f})')
    ax.plot(over[0], over[1], linewidth=1.8,
            label=f'Overdamped (γ = {gamma_over:.2f})')

    ax.plot([x0], [v0], marker='o', color='black', markersize=6,
            linestyle='None', label='Initial state')
    ax.axhline(0, color='black', linewidth=0.6)
    ax.axvline(0, color='black', linewidth=0.6)
    ax.set_xlabel('Position, x')
    ax.set_ylabel('Velocity, v')
    ax.set_title('Phase Diagrams for Three Unforced Damping Regimes')
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()
    plt.close(fig)

free_plot_output = widgets.interactive_output(
    plot_free_phase_diagrams,
    {
        'x0': spos,
        'v0': svel,
        'gamma_under': sgam_under,
        'gamma_over': sgam_over
    }
)

free_reset_button = widgets.Button(description='Reset initial conditions')
def reset_free_values(button):
    spos.value = 5.0
    svel.value = 0.0
    sgam_under.value = 0.5
    sgam_over.value = 1.5
free_reset_button.on_click(reset_free_values)

free_controls = widgets.VBox([spos, svel, sgam_under, sgam_over, free_reset_button])
display(free_controls, free_plot_output)


Output()

## Questions to investigate

### Driven oscillator
1. How does the phase-space trajectory change as the driving frequency approaches the natural frequency?
2. What changes between the early-time trajectory and the later, nearly closed loop?
3. How does stronger damping affect the steady-state response?

### Unforced oscillator
1. Which trajectory spirals toward the origin, and why?
2. Which trajectories approach the origin without oscillating?
3. What happens to the underdamped trajectory as its damping coefficient approaches 1?
4. All three curves begin at the same point. Why do their paths differ afterward?

The phase-space plots show velocity against position, not velocity against time. For the driven oscillator, the transient response decays and the trajectory approaches the steady-state loop. For the unforced oscillator, all three damping regimes approach the origin, although their paths differ.
